# XGBoost connectivity

Paste/run **cell 1** for `/health`. Run **cell 2** only when status is `ok`.

URL: `https://xgboost.derisory.nl` (optional env: `XGBOOST_API_URL`)

In [ ]:
import os
import requests
import urllib3

BASE = os.environ.get("XGBOOST_API_URL", "https://xgboost.derisory.nl").rstrip("/")

# false = skip verify (needed if Traefik serves self-signed cert). Fix prod: use Let's Encrypt on Coolify.
VERIFY_SSL = os.environ.get("XGBOOST_VERIFY_SSL", "false").lower() in ("1", "true", "yes")
if not VERIFY_SSL:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

r = requests.get(f"{BASE}/health", timeout=60, verify=VERIFY_SSL)
print("URL:", f"{BASE}/health")
print("Status:", r.status_code)
print("Body:", r.json())

if r.json().get("status") == "ok":
    print("\nService OK — model loaded. You can call /predict next.")
elif r.json().get("status") == "no_model":
    print("\nService reachable — no model.json on the server yet (predict will 503).")
else:
    print("\nUnexpected health response.")

In [ ]:
# Predict — run only after health shows status "ok"
FEATURES = [0.0, 0.0, 0.0]  # TODO: same length/order as your trained model

r = requests.post(
    f"{BASE}/predict",
    json={"features": FEATURES},
    headers={"Content-Type": "application/json"},
    timeout=60,
    verify=VERIFY_SSL,
)
print("URL:", f"{BASE}/predict")
print("Status:", r.status_code)
print("Body:", r.json() if r.ok else r.text)
r.raise_for_status()